# E3 · Muestra del 100% de las fuentes públicas

**Objetivo.** Evaluar la transferencia de una proyección métrica aprendida con identidades públicas hacia mascotas privadas.

**Pregunta de investigación.** ¿Cómo cambia la recuperación al entrenar la misma MLP con un volumen creciente de datos de Dogs of the World y LCW?

**Hipótesis de trabajo.** La proyección aprendida mejora la recuperación frente a E1/E2. Menor pérdida de entrenamiento o mejor validación pública no demuestran transferencia por sí solas.

La unidad de interés es la **identidad individual de un perro o gato**. La raza no es la etiqueta de recuperación. El análisis forma parte de una secuencia incremental: E1 establece la línea base, E2 incorpora localización y E3 aprende una proyección métrica.

| Elemento | Diseño de E3 |
|---|---|
| Entrada | El mismo pipeline de detección y recorte de E2 |
| Representación | E2 → MLP 512→256→512 → normalización L2 |
| Parámetros ajustados | Solo la MLP; YOLO y CLIP permanecen congelados |
| Entrenamiento y selección | Identidades públicas para entrenamiento; checkpoint por mAP macro de validación pública |
| Artefacto | E2 como base; artefacto E3 exportado por la corrida |
| Control | E1/E2 congelados: resultados publicados sobre las mismas consultas y galerías |
| Evaluación | Test público v1 inmutable y PetTrace v4, por separado |
| Volumen solicitado | 100% de las fotos originales de **cada** fuente, antes de limpieza; no del piloto v1 |
| Indicadores | Rank-1/2/5/10, mAP por fotografía y EER descriptiva; resultados globales y por especie |
| Trazabilidad | Publicación obligatoria en MLflow; registro de versiones del modelo separado |

La corrida `20261030T153523Z-e258882a` conserva entrenamiento y evaluación reales en las salidas de este notebook. Su salida histórica indica publicación desactivada en ese momento; la política actual exige publicarla y verificar su recibo. El benchmark v4 ya fue observado en E1/E2, por lo que el contraste se interpreta como exploratorio.

**Estado de esta variante:** no hay resultados nuevos del 100%. La muestra ampliada debe prepararse y versionarse antes de entrenar. El piloto v1 no satisface este volumen.


### Entorno de ejecución y procedencia

Instala el wheel **v0.2.4** con el perfil E3 de Linux/Python 3.13. Usa los archivos construidos del paquete: wheel, `SHA256SUMS` y `requirements-e3-runtime.txt`. El setup conserva el kernel de Colab y solo solicita reiniciar si cambió una biblioteca ya cargada. Esta versión requiere publicar la nueva release para distribución por URL; no se descarga una release inexistente.

E1/E2 permanecen congelados. E3 usa PyTorch 2.9.1 y ONNX 1.19.1, por lo que las salidas históricas del piloto no acreditan resultados del nuevo entorno. Guarda versiones, GPU, semillas y volúmenes reales con cada corrida.


In [ ]:
# Upload the built wheel, SHA256SUMS and requirements-e3-runtime.txt to a fresh Colab runtime.
import hashlib
import subprocess
import sys
from importlib import metadata
from pathlib import Path

if sys.version_info[:2] != (3, 13) or sys.platform != "linux":
    raise RuntimeError("This E3 Colab profile requires Linux and Python 3.13")
requirements = next((p for p in (Path("requirements-e3-runtime.txt"), Path("requirements/e3-runtime.txt")) if p.is_file()), None)
wheel = Path("pettrace_ml-0.2.4-py3-none-any.whl")
checksums = Path("SHA256SUMS")
if requirements is None or not wheel.is_file() or not checksums.is_file():
    raise FileNotFoundError("Place the v0.2.4 wheel, SHA256SUMS and requirements-e3-runtime.txt beside this notebook in the runtime")
if hashlib.sha256(requirements.read_bytes()).hexdigest() != "906d040f5ea4881e4750b553908afd65cf3bc88a64d8e741951e9f9ffd64aa3a":
    raise ValueError("E3 requirements checksum mismatch")
expected = {name: digest for digest, name in (line.split() for line in checksums.read_text().splitlines() if line.strip())}
if hashlib.sha256(wheel.read_bytes()).hexdigest() != expected.get(wheel.name):
    raise ValueError("E3 wheel checksum mismatch")
loaded = {name.split(".")[0] for name in sys.modules}
before = {dist: metadata.version(dist) for module, distributions in metadata.packages_distributions().items()
          if module in loaded for dist in distributions}
install_log = Path("e3-install.log")
with install_log.open("w") as log:
    for args in (["--require-hashes", "-r", str(requirements)], ["--no-deps", str(wheel)]):
        result = subprocess.run([sys.executable, "-m", "pip", "install", *args], stdout=log, stderr=subprocess.STDOUT)
        if result.returncode:
            raise RuntimeError(f"Installation failed; inspect {install_log}")
_E3_RESTART_REQUIRED = sorted(set(globals().get("_E3_RESTART_REQUIRED", [])) |
                              {dist for dist, old in before.items() if metadata.version(dist) != old})
print("Restart this session, then continue at verification." if _E3_RESTART_REQUIRED else "Installed; continue at verification.")


In [ ]:
if globals().get("_E3_RESTART_REQUIRED"):
    raise RuntimeError("Restart the session, then continue from this verification cell")
import hashlib
from pathlib import Path
from importlib.metadata import version
from packaging.requirements import Requirement
requirements = next(p for p in (Path("requirements-e3-runtime.txt"), Path("requirements/e3-runtime.txt")) if p.is_file())
if hashlib.sha256(requirements.read_bytes()).hexdigest() != "906d040f5ea4881e4750b553908afd65cf3bc88a64d8e741951e9f9ffd64aa3a":
    raise ValueError("E3 requirements checksum mismatch")
for line in requirements.read_text().splitlines():
    if line and not line[0].isspace() and not line.startswith("#"):
        req = Requirement(line.rstrip(chr(92) + " "))
        if (req.marker is None or req.marker.evaluate()) and version(req.name) not in req.specifier:
            raise RuntimeError(f"{req.name} differs from the E3 runtime lock")
import os
import importlib.util

# Import torch before ONNX when installed in the runtime.
if importlib.util.find_spec("torch") is not None:
    import torch
import pettrace
from pettrace.tracking import source_identity

if pettrace.__version__ != "0.2.4":
    raise RuntimeError("Install release v0.2.4 and restart the kernel before continuing")
code_identity = source_identity()
if code_identity["build"].get("version") != "0.2.4":
    raise RuntimeError("Use the published release wheel to retain its commit and lockfile provenance")
SOURCE_SHA256 = code_identity["source_sha256"]
print("Package:", pettrace.__version__)
print("Commit:", code_identity["git_sha"], "· source hash:", SOURCE_SHA256)
print("Dependency lock:", code_identity["build"]["uv_lock_sha256"])

from pettrace.data.volume import training_volume
from pettrace.experiments.e3 import frozen_controls


In [ ]:
# Request access only after verifying the prepared dataset; never print credential values.
def configure_private_access(names=("HF_TOKEN", "MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD")):
    # Credentials stay in environment variables / Secrets / hidden prompts, never in notebook source.
    import getpass
    import os
    import warnings

    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    for name in names:
        if not os.environ.get(name) and userdata is not None:
            try:
                os.environ[name] = userdata.get(name)
            except Exception:
                pass  # VS Code may not expose Secrets; use a hidden prompt below.
        if not os.environ.get(name):
            if name == "MLFLOW_TRACKING_URI":
                value = "https://ml.pettrace.app"
            else:
                # Fail before reading input if the kernel cannot hide the entered value.
                with warnings.catch_warnings():
                    warnings.simplefilter("error", getpass.GetPassWarning)
                    value = getpass.getpass(f"{name}: ")
            if not value.strip():
                raise ValueError(f"Missing {name}.")
            os.environ[name] = value


In [ ]:
import json
import hashlib
import os
import shutil
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import mlflow
import numpy as np
import torch
import onnxruntime as ort
from IPython.display import Image, Markdown, display
from huggingface_hub import HfApi, snapshot_download

from pettrace.data import benchmark as data
from pettrace.training.triplet import Config, fit, export_encoder
from pettrace.data.preparation import load_prepared
from pettrace.evaluation.core import evaluate, load_model
from pettrace.evaluation.figures import save
from pettrace.training.features import feature_cache
from pettrace.data.curation import sha256
from pettrace.tracking import PRODUCTION_URI, publish_evaluation, save_provenance, source_identity

EXPERIMENT = "E3"
SOURCE_SAMPLE_PCT = 100
DATASET = "v4"
PROTOCOL = "zero-shot"
DATASET_COMMIT = "8b12db0dce41040a821f9792bea5b352342263ce"
BASE_MODEL = "models:/pettrace-embedder/2"
BASELINE_MODEL = "models:/pettrace-embedder/1"
ENCODER_SHA256 = "4773e649274ca8774c4a63d45f0438fe598ee7ba67e65def97537aa0eca9a576"
DETECTOR_SHA256 = "da8f9b9ebea5157cdcefa7bea9fd782249ebdf119f917c07d64e5b41942b9c9d"
LOG_TO_MLFLOW = True
REGISTER_E3 = True
SEED = 42
DATA_ROOT = Path(os.environ.get("E3_DATA_ROOT", "datasets/public/e3-body"))
PREPARED_DATASET_ROOT = Path(os.environ.get("E3_PREPARED_DATASET", str(DATA_ROOT / f"source-{SOURCE_SAMPLE_PCT}/prepared/latest.json")))
TRAINING_DATASET_REPO = os.environ.get("HF_TRAINING_DATASET_REPO", "alessandro54/pettrace-training")
TRAINING_DATASET_VERSION = os.environ.get("HF_TRAINING_DATASET_VERSION", "")
TRAINING_DATASET_REVISION = os.environ.get("HF_TRAINING_DATASET_REVISION", "")
TRAINING_DATASET_FINGERPRINT = os.environ.get("HF_TRAINING_DATASET_FINGERPRINT", "")
FEATURE_CACHE_ROOT = Path(os.environ.get("E3_FEATURE_CACHE", str(DATA_ROOT / "feature-cache")))
RUN_LABEL = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
REPORT = Path(os.environ.get("PETTRACE_OUTPUT_DIR", "reports")) / f"e3_{SOURCE_SAMPLE_PCT}_{DATASET}" / RUN_LABEL
REPORT.mkdir(parents=True, exist_ok=False)
(REPORT / "requirements-e3-runtime.txt").write_bytes(requirements.read_bytes())
SOURCE_IDENTITY = source_identity()
SOURCE_SHA256 = SOURCE_IDENTITY["source_sha256"]
mlflow.set_tracking_uri(PRODUCTION_URI)
mlflow.set_registry_uri(PRODUCTION_URI)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Package source:", SOURCE_IDENTITY)
print("Training device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Training source:", TRAINING_DATASET_REPO or PREPARED_DATASET_ROOT)
if TRAINING_DATASET_REPO:
    print("Training revision:", TRAINING_DATASET_REVISION)
print("Requested source sample:", SOURCE_SAMPLE_PCT, "% of each original source")
print("Report:", REPORT)
print("Publish to MLflow:", LOG_TO_MLFLOW, "| Register model:", REGISTER_E3)
assert device == "cuda", "Select a GPU runtime (T4 is sufficient), then rerun this notebook"


## Fase 1. Preparación y preprocesamiento de datos

La muestra pública procede de `pettrace-training`, preparado de forma independiente en `datasets/prepare_training.ipynb`. Puede reutilizarse en otros experimentos. La versión conserva fotos, splits por individuo, reglas de limpieza, exclusiones y hashes; el experimento verifica su integridad antes de entrenar.

Esta variante solicita **100% de cada archivo fuente original**, antes de limpiar o dividir. Configura `HF_TRAINING_DATASET_VERSION`, `HF_TRAINING_DATASET_REVISION` (commit inmutable) y `HF_TRAINING_DATASET_FINGERPRINT` para la versión ampliada; o `HF_TRAINING_DATASET_REPO=""` y `E3_PREPARED_DATASET` para una copia local verificada. El piloto v1 se conserva como referencia histórica y el control de volumen lo rechaza: sus 1.276 fotos de train no representan el 20% de las fuentes.

Los conteos solicitados se redondean hacia abajo por fuente. La preparación debe conservar los recibos de adquisición con las rutas y hashes de todas las fotos muestreadas, incluso las descartadas después. Se mantienen las mismas identidades/fotos de validación v1 en las tres variantes y se excluyen del entrenamiento todas las identidades del test público v1. No se adquieren ni limpian datos en este notebook. La adquisición ampliada es un requisito pendiente, no un resultado existente.

`pettrace-reid@v4` mantiene su función de benchmark privado; no aporta fotos al dataset de entrenamiento.

La adquisición, limpieza, deduplicación, filtro YOLO y asignación de splits de las fuentes públicas se documentan en [`datasets/prepare_training.ipynb`](../datasets/prepare_training.ipynb). Esta fase consume la versión preparada; no vuelve a ejecutar esa preparación dentro del experimento. La curación/exportación de PetTrace v4 constituye un flujo privado separado. Las reglas de muestreo y filtrado son decisiones de esta investigación, no resultados atribuidos a la literatura.


In [ ]:
from pathlib import Path
if Path("/content").is_dir():
    from pettrace.data.checkpoints import DriveBackup
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive
        drive.mount("/content/drive")
    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Mount Drive before restoring prepared data")
    preparation_backup = DriveBackup({
        "data": DATA_ROOT,
        "prepared": Path("/content/datasets/build/pettrace-training"),
        "reference": Path("/content/datasets/build/hub-training"),
    }, Path("/content/drive/MyDrive/PetTrace/checkpoints/preparation"))
    preparation_backup.restore(only={"prepared"})


from pettrace.data.versions import load_local_training, load_training

if Path("/content").is_dir():
    local_version = Path(f"/content/datasets/build/pettrace-training/v2.{SOURCE_SAMPLE_PCT}.json")
    if local_version.is_file() and not TRAINING_DATASET_VERSION:
        PREPARED_DATASET_ROOT = local_version
        TRAINING_DATASET_REPO = ""
if TRAINING_DATASET_REPO:
    import re
    if not re.fullmatch(r"[0-9a-f]{40}", TRAINING_DATASET_REVISION) or not TRAINING_DATASET_FINGERPRINT:
        raise ValueError("Prepare and version the requested source sample first; set its immutable commit and fingerprint")
    configure_private_access(names=("HF_TOKEN",))
    prepared, training_source = load_training(
        repo_id=TRAINING_DATASET_REPO, version=TRAINING_DATASET_VERSION, revision=TRAINING_DATASET_REVISION,
        expected_fingerprint=TRAINING_DATASET_FINGERPRINT)
else:
    prepared, training_source = load_local_training(PREPARED_DATASET_ROOT)
manifest_rows = prepared.rows
manifest_path = prepared.root / "manifest.csv"
review_path = prepared.root / "review.csv"
audit_summary = json.loads((prepared.root / "audit.json").read_text())
detection_audit = json.loads((prepared.root / "animal-screening.json").read_text())
for filename in ("manifest.csv", "review.csv", "audit.json", "sources.json", "animal-screening.json", "prepared.json"):
    shutil.copy2(prepared.root / filename, REPORT / filename)
(REPORT / "data-preparation.json").write_text(json.dumps(
    {"prepared_dataset_fingerprint": prepared.fingerprint, "training_source": training_source,
     "root": str(prepared.root),
     "prepared_receipt_sha256": sha256(prepared.root / "prepared.json"),
     "manifest_sha256": sha256(manifest_path), "counts": prepared.summary["counts"]}, indent=2))
print("Prepared dataset fingerprint:", prepared.fingerprint)
print("Photos:", len(manifest_rows), "| Identities:", prepared.summary["retained_identities"])
print(json.dumps(prepared.summary["counts"], indent=2))
print(prepared.summary["limitation"])


### 1.1 Cobertura de las fuentes originales

La escala de la corrida se expresa como **fotos utilizadas / fotos originales × 100**. El denominador queda fijado por versión y hash del índice ZIP: Dogs of the World v1 contiene 313.671 entradas de imagen bajo `images/`; LCW v1 contiene 380.923 bajo `CatFullDataset/<ID>/`. En conjunto son 694.594 fotos antes de limpieza. No equivalen al máximo de fotos elegibles para reidentificación: pueden incluir individuos con pocas tomas, duplicados o escenas excluidas.

`dataset.train_pct` mide únicamente fotos que ajustan pesos; `dataset.train_val_pct` incorpora la validación. Los porcentajes por especie se conservan por separado. El test público continúa reservado. Estos valores quedan en `dataset-coverage.json` y se publican obligatoriamente como métricas consultables de MLflow, junto con la procedencia del denominador.

**No son el mismo porcentaje:** `dataset.sample_requested_pct` es la variante 100%; `dataset.sampled_pct` cuenta las fotos adquiridas de los archivos originales; `dataset.train_pct` cuenta solo las fotos que actualizan pesos después de limpiar y separar. Se registran también fotos e identidades de train/val, globales y por especie. El porcentaje adquirido se verifica por fuente, no solo en el total combinado.


In [ ]:
from pettrace.data.volume import training_volume
coverage_metrics, coverage_params = training_volume(prepared, SOURCE_SAMPLE_PCT, REPORT)
print(f"Processed: {coverage_metrics['dataset.processed_photos']:,} photos ({coverage_metrics['dataset.processed_pct']:.4f}% of sources)")
volume_table = ["| Fuente | Originales | Muestreadas | % adquirido | Train fotos | Train identidades | % fuente en train |", "|---|---:|---:|---:|---:|---:|---:|"]
for label, prefix in (("Dogs of the World", "dataset.dog"), ("LCW", "dataset.cat"), ("Total", "dataset")):
    volume_table.append(f"| {label} | {coverage_metrics[prefix+'.source_photos']:,} | {coverage_metrics[prefix+'.sampled_photos']:,} | "
                        f"{coverage_metrics[prefix+'.sampled_pct']:.4f}% | {coverage_metrics[prefix+'.train_photos']:,} | "
                        f"{coverage_metrics[prefix+'.train_identities']:,} | {coverage_metrics[prefix+'.train_pct']:.4f}% |")
display(Markdown("\n".join(volume_table)))


### 1.2 Benchmark privado fijado a v4

La descarga privada se fija al mismo commit de E1/E2 v4. Ninguna mascota privada participa de la preparación pública, el entrenamiento o la selección del checkpoint. Se comprueba el solapamiento exacto de imágenes con el dataset preparado.


In [ ]:
configure_private_access()
resolved_commit = HfApi().dataset_info(data.REPO, revision=DATASET).sha
assert resolved_commit == DATASET_COMMIT, "The v4 tag changed; review the comparison protocol"
private_root = snapshot_download(data.REPO, repo_type="dataset", revision=DATASET_COMMIT)
ds = data.from_dir(private_root, repo=data.REPO, version=DATASET, commit=DATASET_COMMIT)
private_hashes = {sha256(photo.path) for photo in ds.photos}
assert not private_hashes.intersection(row["sha256"] for row in manifest_rows), "Public/private image overlap"
print("Benchmark:", ds.name, "| Commit:", ds.commit, "| Photos:", len(ds.photos))
for species in ("cat", "dog"):
    photos = [photo for photo in ds.photos if photo.species == species]
    print(species, "identities:", len({photo.animal for photo in photos}), "photos:", len(photos))


### 1.3 Test público reservado y comparación entre dominios

El test público procede de **Dogs of the World y LCW (Kaggle)**, mediante la versión preparada `pettrace-training@v1`. Se conservan sus **277 fotos y 60 identidades** (34 gatos y 26 perros), sin mezclar train/val ni volver a dividir las imágenes. Representa el **15,2 % de las fotos y 14,9 % de las identidades preparadas**, no el 20 %. Se mantienen el commit y fingerprint de v1.

El experimento evalúa dos galerías separadas: este test público y PetTrace v4. La diferencia entre sus métricas es descriptiva: cambian individuos, cantidad de candidatos y condiciones de captura. No se combinan las fotos ni se atribuye causalmente la diferencia a Kaggle o PetTrace. El test público deja de ser una referencia nunca observada después de su primera evaluación; no se usará para seleccionar el checkpoint de E3.


In [ ]:
from pettrace.evaluation.research import public_test_dataset, PUBLIC_REPO, PUBLIC_VERSION, PUBLIC_COMMIT, PUBLIC_FINGERPRINT
from pettrace.data.versions import load_training
from pettrace.data.curation import sha256

public_prepared, public_source = load_training(
    repo_id=PUBLIC_REPO, version=PUBLIC_VERSION, revision=PUBLIC_COMMIT,
    expected_fingerprint=PUBLIC_FINGERPRINT)
# Expanded source samples must not move the original validation or test identities.
public_test_rows = [row for row in public_prepared.rows if row["split"] == "test"]
public_val_rows = [row for row in public_prepared.rows if row["split"] == "val"]
training_rows = [row for row in manifest_rows if row["split"] in {"train", "val"}]
if ({row["animal"] for row in training_rows} & {row["animal"] for row in public_test_rows}
        or {row["sha256"] for row in training_rows} & {row["sha256"] for row in public_test_rows}):
    raise ValueError("Expanded train/val overlaps the immutable public test")
validation_signature = lambda rows: sorted((row["animal"], row["species"], row["sha256"]) for row in rows)
if validation_signature([row for row in training_rows if row["split"] == "val"]) != validation_signature(public_val_rows):
    raise ValueError("Keep the original v1 validation photos and identities fixed across source volumes")
public_ds = public_test_dataset(public_prepared, public_source)
public_hashes = {row["sha256"] for row in public_prepared.rows}
if public_hashes.intersection(sha256(photo.path) for photo in ds.photos):
    raise ValueError("An exact image is shared by the public source and PetTrace benchmark")
PUBLIC_REPORT = REPORT / "public_test"
PUBLIC_REPORT.mkdir(parents=True, exist_ok=True)
selection = {"scope": "published_test_only", "source": public_source,
             "prepared_fingerprint": public_prepared.fingerprint,
             "evaluation_fingerprint": public_ds.fingerprint,
             "photos": len(public_ds.photos), "identities": len(public_ds.animals),
             "rows": [{"animal": row["animal"], "species": row["species"], "sha256": row["sha256"]}
                      for row in public_prepared.rows if row["split"] == "test"]}
(PUBLIC_REPORT / "dataset-selection.json").write_text(json.dumps(selection, indent=2))
print("Public test:", len(public_ds.photos), "photos |", len(public_ds.animals), "identities")
print("Public commit:", public_ds.commit, "| Test fingerprint:", public_ds.fingerprint)


## Fase 2. Representación y entrenamiento

El entrenamiento reutiliza E2 sin actualizar detector ni encoder. Sus hashes y golden set se verifican de nuevo, independientemente del detector empleado en la preparación. Solo se ajusta la cabeza MLP.

**Fundamento externo.** El encoder procede de [Radford et al. (2021), CLIP](https://arxiv.org/abs/2103.00020). Su aplicación a identidades de mascotas y los controles de este cuaderno corresponden a esta investigación.
El detector se basa en [Ultralytics, YOLOv8](https://docs.ultralytics.com/models/yolov8/); el margen, la política de selección de caja y su auditoría se especifican como decisiones del proyecto.


In [ ]:
base_model = load_model(BASE_MODEL)
assert base_model.golden_ok, "E2 golden validation failed"
assert base_model.sha256 == {"encoder.onnx": ENCODER_SHA256, "detector.onnx": DETECTOR_SHA256}
assert base_model.pipeline.detector is not None, "E3 requires the E2 crop pipeline"
print("Base:", base_model.uri, "| Golden:", base_model.golden_fingerprint)


### 2.1 Ajuste de parámetros y selección

La cabeza MLP transforma vectores de 512 dimensiones mediante una capa intermedia de 256, ReLU y normalización L2 a la salida. La pérdida acerca ejemplos de la misma identidad y separa identidades distintas:

$$\mathcal{L}=\frac{1}{|A|}\sum_{a\in A}\max\left(0,\max_{p\ne a,\ y_p=y_a}d(z_a,z_p)-\min_{n:\ y_n\ne y_a}d(z_a,z_n)+m\right),\quad m=0.2.$$

El muestreo P×K reúne 8 identidades y 4 imágenes por identidad, alternando lotes de gatos y perros. Los positivos y negativos se seleccionan dentro del lote y de la misma especie. La existencia de distintas fotos no certifica que procedan de sesiones diferentes.

La configuración implementada utiliza AdamW, tasa de aprendizaje 1e-3, weight decay 1e-4, hasta 30 épocas y paciencia 7. Los embeddings de E2 se cachean porque el extractor no cambia. El checkpoint se selecciona exclusivamente con mAP macro por especie de la validación pública; la muestra privada no selecciona parámetros ni época.

La exportación fusiona encoder y cabeza en ONNX y verifica el artefacto resultante. Estas verificaciones controlan la implementación, sin sustituir una corrida real ni demostrar mejora de recuperación.

La inicialización y el muestreo usan semilla 42. Antes de entrenar, el reporte conserva la configuración, el manifiesto revisado, el hash de caché y la identidad exacta del extractor. Las curvas de pérdida y validación se muestran después del ajuste.

**Referencia del aprendizaje métrico.** La selección batch-hard se fundamenta en [Hermans, Beyer y Leibe (2017), *In Defense of the Triplet Loss for Person Re-Identification*](https://arxiv.org/abs/1703.07737). La arquitectura MLP, el muestreo alternado por especie, los hiperparámetros y el criterio de checkpoint son decisiones de esta investigación; los resultados del artículo en personas no se presentan como evidencia en mascotas.

**Control de volumen.** Las tres variantes mantienen semilla, arquitectura, P×K, validación y máximo de 100 pasos por época. Una época es un presupuesto fijo de pasos, no una pasada completa por todas las fotos; el volumen disponible y la exposición efectiva se interpretan por separado.


In [ ]:
config = Config(seed=SEED, epochs=30, steps_per_epoch=100, identities_per_batch=8,
                images_per_identity=4, hidden=256, margin=0.2, learning_rate=1e-3,
                weight_decay=1e-4, patience=7)
assert device == "cuda", "Select a GPU runtime for the real E3 training run"
training_rows = [row for row in manifest_rows if row["split"] in {"train", "val"}]
print("Extracting / reusing frozen E2 embeddings:", len(training_rows), "train/val photos", flush=True)
from pettrace.data.checkpoints import DriveBackup
if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Mount Drive before extracting reusable embeddings")
feature_backup = DriveBackup({"features": FEATURE_CACHE_ROOT},
    Path("/content/drive/MyDrive/PetTrace/checkpoints/features"))
feature_backup.restore()
with feature_backup:
    features, cache_digest = feature_cache(base_model.pipeline, training_rows, FEATURE_CACHE_ROOT,
                                          base_model.dir, checkpoint=feature_backup)
from dataclasses import asdict
run_config = {"experiment": EXPERIMENT, "source_sample_pct": SOURCE_SAMPLE_PCT, "dataset_coverage": coverage_metrics, "seed": SEED, "training": asdict(config),
              "prepared_dataset_fingerprint": prepared.fingerprint, "training_dataset_source": training_source,
              "dataset": ds.name, "dataset_commit": ds.commit, "dataset_fingerprint": ds.fingerprint,
              "base_model": base_model.uri, "base_model_sha256": base_model.sha256,
              "cache_sha256": cache_digest, "manifest_sha256": sha256(REPORT / "manifest.csv"),
              "review_sha256": sha256(REPORT / "review.csv"),
              "animal_screening_sha256": sha256(REPORT / "animal-screening.json"), "source": SOURCE_IDENTITY,
              "runtime_lock_sha256": hashlib.sha256(requirements.read_bytes()).hexdigest(), "log_to_mlflow": LOG_TO_MLFLOW, "register_model": REGISTER_E3}
(REPORT / "training-config.json").write_text(json.dumps(run_config, indent=2))
report_backup = DriveBackup({"report": REPORT},
    Path("/content/drive/MyDrive/PetTrace/checkpoints/runs") / REPORT.name)

def show_epoch(record):
    report_backup()
    print(f"Epoch {record['epoch']:02d}/{config.epochs} | loss {record['train.loss']:.4f} | "
          f"validation macro mAP {record['val.macro_species_mAP']:.4f}", flush=True)

print("Training MLP on", device, "| public test excluded | MLflow publication:", LOG_TO_MLFLOW, flush=True)
with report_backup:
    trained_head, train_summary = fit(features, training_rows, REPORT, config=config, device=device,
                                      callback=show_epoch)
    (REPORT / "training-summary.json").write_text(json.dumps(train_summary, indent=2))
    print("Selected epoch:", train_summary["best_epoch"], "| Public validation macro mAP:", train_summary["best_val_mAP"])


In [ ]:
import matplotlib.pyplot as plt
history = train_summary["history"]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].plot([row["epoch"] for row in history], [row["train.loss"] for row in history])
axes[0].set(xlabel="Época", ylabel="Triplet loss", title="Optimización pública")
axes[1].plot([row["epoch"] for row in history], [row["val.macro_species_mAP"] for row in history])
axes[1].axvline(train_summary["best_epoch"], color="gray", linestyle="--")
axes[1].set(xlabel="Época", ylabel="mAP macro", title="Selección en validación pública")
fig.tight_layout()
fig.savefig(REPORT / "training-curves.png", dpi=160)
plt.show()


In [ ]:
model_dir = REPORT / "model"
onnx_hashes = export_encoder(base_model.dir, trained_head, model_dir, model_version=f"clip-vit-b32-crop-mlp-source{SOURCE_SAMPLE_PCT}@v1")
print("Exported E3 artifacts:", onnx_hashes)


## Fase 3. Evaluación y análisis de resultados

El artefacto fusionado E3 se evalúa sobre `pettrace-reid@v4` mediante el evaluador compartido y una reconstrucción visible del ranking. Se excluye la propia foto de la galería, se restringen candidatos a la misma especie y se utiliza el máximo score por identidad para Rank-k. AP/mAP conserva el ranking por fotografías.

La evaluación usa todas las fotos elegibles de v4. E1 y E2 se recalculan dentro del notebook sobre esa misma muestra. Las líneas base se recalculan sobre el mismo objeto dataset y sus consultas se comparan explícitamente.

La EER mostrada se ajusta sobre los mismos pares privados en los que se reporta, por lo que es descriptiva. No representa calibración independiente ni un umbral validado para despliegue. La nueva ejecución evalúa además el test público reservado de v1, con su propia galería; sus resultados no seleccionan el checkpoint. El runner extendido conserva otro alcance documentado en el protocolo de E3.

#### 3.1 Alcance común de la recuperación

La evaluación es **de galería cerrada**: la identidad de cada consulta evaluable tiene otra fotografía en la galería. Se excluye la propia foto y se restringen candidatos a su especie anotada. Por tanto, los resultados están condicionados a conocer la especie y no miden detección de identidades ausentes, búsqueda en una galería abierta ni rendimiento de un sistema de mascotas perdidas en producción.

La etiqueta técnica `zero-shot` del evaluador activa este protocolo sobre todas las fotos elegibles de v4. No implica clasificación con prompts de texto. En E3 existe entrenamiento con identidades públicas, pero ninguna identidad privada se usa para actualizar pesos o seleccionar checkpoint. La separación consulta/galería es por fotografía y no garantiza sesiones independientes.


#### 3.1.3 Definición de indicadores y límites de la verificación

Rank-k mide el porcentaje de consultas que recuperan el animal correcto entre los k primeros; mAP resume el ranking de fotos. El promedio global pondera por consultas. El azar uniforme entre animales es `min(k / animales_en_galería, 1)` para cada consulta y luego se promedia; así no se presupone que gatos y perros tengan el mismo tamaño de galería.

EER usa pares no ordenados, sin diagonal, de la misma especie con al menos dos animales. Se ajusta sobre todos los pares evaluados, incluyendo todos los splits. No es calibración independiente en validación, aunque el esquema compartido almacene estos valores bajo `val`.

Para $Q$ consultas evaluables, con $r_i$ como posición del animal correcto:

$$\mathrm{Rank}@k = \frac{1}{Q}\sum_{i=1}^{Q}\mathbf{1}[r_i\le k].$$

Si $P_i(j)$ es la precisión hasta la posición $j$ del ranking de fotos y $R_i$ es el número de fotos relevantes de la consulta:

$$\mathrm{AP}_i=\frac{1}{R_i}\sum_j P_i(j)\,\mathbf{1}[\text{foto}_j\text{ del mismo animal}],\qquad
\mathrm{mAP}=\frac{1}{Q}\sum_i\mathrm{AP}_i.$$

Rank-k usa el máximo score por animal; AP conserva las fotos individuales. Un promedio macro por identidad asignaría el mismo peso a cada mascota; no se confunde aquí con el promedio por consultas reportado.

La EER se aproxima en los umbrales observados buscando la menor diferencia entre FAR (pares de animales distintos aceptados) y FRR (pares del mismo animal rechazados). Como se ajusta y reporta sobre los mismos pares, no valida un umbral de producción.

**Referencia y adaptación.** CMC y AP/mAP se emplean en reidentificación, por ejemplo en [Zheng et al. (2015), *Scalable Person Re-Identification: A Benchmark*](https://openaccess.thecvf.com/content_iccv_2015/html/Zheng_Scalable_Person_Re-Identification_ICCV_2015_paper.html). Esta investigación implementa su propio protocolo de galería por especie y máximo score por identidad; no reproduce el protocolo de cámaras de Market-1501. Rank-2 aplica la misma definición Rank-k con k=2. Las tasas de error de coincidencia se contextualizan en [NIST (2022), *A Tale of Two Errors*](https://www.nist.gov/blogs/taking-measure/tale-two-errors-measuring-biometric-algorithms); el cálculo discreto de EER aquí es una decisión de implementación y no una calibración avalada por NIST.


In [ ]:
e3_model = load_model(str(model_dir))
assert e3_model.golden_ok, "El modelo E3 fusionado no superó la verificación golden."
ev = evaluate(ds, e3_model.pipeline, zero_shot=PROTOCOL == "zero-shot")
E = ev.embeddings
print(f"Embeddings calculados: {E.shape}, norma media = {np.linalg.norm(E, axis=1).mean():.4f}")


In [ ]:
animals = np.array([p.animal for p in ds.photos])
species = np.array([p.species for p in ds.photos])
S = E @ E.T  # Similitud coseno entre todas las fotos

# Evaluación leave-one-out dentro de la misma especie
rows_eval_calc = []
for q in range(len(E)):
    gallery = np.where((species == species[q]) & (np.arange(len(E)) != q))[0]
    if len(set(animals[gallery])) < 2 or not (animals[gallery] == animals[q]).any():
        continue
    sub = S[q, gallery]
    sub_animals = animals[gallery]
    sub_species = species[gallery]

    unique_ids = []
    seen = set()
    for a in sub_animals:
        if a not in seen:
            seen.add(a)
            unique_ids.append(a)
    scores = np.array([sub[sub_animals == a].max() for a in unique_ids])
    order = np.argsort(-scores, kind="stable")
    ordered_ids = np.array(unique_ids)[order]
    rank = int(np.where(ordered_ids == animals[q])[0][0]) + 1

    # Average Precision
    photo_order = np.argsort(-sub, kind="stable")
    labels = (sub_animals[photo_order] == animals[q]).astype(int)
    k = np.arange(1, len(labels) + 1)
    prec = np.cumsum(labels) / k
    ap = float((prec * labels).sum() / max(1, labels.sum()))

    rows_eval_calc.append({"query": q, "animal": animals[q], "species": species[q], "rank": rank, "ap": ap})

ranks = np.array([r["rank"] for r in rows_eval_calc])
aps = np.array([r["ap"] for r in rows_eval_calc])

print("Resultados calculados a mano:")
print(f"  Rank-1: {np.mean(ranks <= 1)*100:.1f} %")
print(f"  Rank-5: {np.mean(ranks <= 5)*100:.1f} %")
print(f"  mAP   : {aps.mean():.3f}")


In [ ]:
expected_ranks = {row["query"]: row["rank"] for row in ev.result["test"]["per_query"]}
assert all(row["rank"] == expected_ranks[row["query"]] for row in rows_eval_calc)
assert np.isclose(aps.mean(), ev.result["test"]["global"]["mAP"])
print("Verificación del cálculo visible con pettrace.evaluation.core: OK")


### 3.2 Resultados

**Evidencia histórica del piloto**, no de la variante 100%. Las tablas calculadas después de ejecutar esta variante serán su evidencia; no se atribuyen estas cifras al nuevo volumen.

La corrida conservada evaluó 305 consultas de 31 mascotas privadas (15 gatos y 16 perros). La tabla de referencia resume las salidas guardadas; las celdas siguientes evalúan E3 y leen E1/E2 publicados para el mismo snapshot y verifican que sus consultas y galerías coincidan.

| Modelo | Rank-1 | Rank-5 | mAP | EER descriptiva |
|---|---:|---:|---:|---:|
| E1 | 65,90 % | 94,10 % | 0,3751 | 36,23 % |
| E2 | 67,87 % | 92,13 % | 0,3960 | 33,83 % |
| E3 | 72,79 % | 96,39 % | 0,5682 | 16,76 % |

Frente a E2, E3 aumenta Rank-1 en 4,92 puntos porcentuales y mAP en 0,1721 según el delta calculado con precisión completa. En gatos, Rank-1 cambia de 67,31 % a 76,28 %; en perros, de 68,46 % a 69,13 %. mAP mejora en ambas especies: 0,3950→0,6171 en gatos y 0,3971→0,5170 en perros. El cambio en el primer candidato es, por tanto, desigual entre especies.

El checkpoint seleccionado corresponde a la época 3, con mAP macro de validación pública 0,6603. La ejecución se detuvo en la época 10 tras siete épocas sin superar ese valor. Las curvas públicas documentan optimización y selección; las métricas privadas documentan transferencia. Se trata de una sola semilla, sin intervalos de confianza ni contraste de significancia.


In [ ]:
from pettrace.evaluation.research import save_benchmark_comparison

public_ev = evaluate(public_ds, e3_model.pipeline, zero_shot=True)
public_summary = save(public_ev, e3_model, str(PUBLIC_REPORT))
save_provenance(PUBLIC_REPORT, e3_model)
comparison_markdown, comparison_figure = save_benchmark_comparison(public_ev, ev, REPORT, EXPERIMENT)
display(Markdown(comparison_markdown))
display(comparison_figure)
import matplotlib.pyplot as plt
plt.close(comparison_figure)
for filename, caption in (("confusion.png", "Confusión top-1 del test público reservado, por identidad y especie."),
                          ("cmc.png", "CMC del test público: incluye Rank-1, Rank-2 y otros rangos."),
                          ("rank1_rank2_map.png", "Rank-1/Rank-2 y mAP públicos, globales y por especie.")):
    display(Image(str(PUBLIC_REPORT / filename), width=900))
    display(Markdown(caption))

from pettrace.experiments.e3 import frozen_controls
from pettrace.evaluation.figures import comparison_figure as plot_comparison
public_controls, public_control_summaries = frozen_controls(public_ev, "public", PUBLIC_REPORT)
fig = plot_comparison({**public_controls, "E3": public_ev}, "Test público · controles congelados")
fig.savefig(PUBLIC_REPORT / "frozen-comparison.png", dpi=160, bbox_inches="tight")
display(fig)
plt.close(fig)
public_deltas = {label: {key: public_summary["global"][key]-summary["global"][key]
                       for key in ("rank1", "rank5", "mAP")} for label, summary in public_control_summaries.items()}
(PUBLIC_REPORT / "comparison.json").write_text(json.dumps(public_deltas, indent=2))
print("E3 menos cada control publicado:", public_deltas)


In [ ]:
from pettrace.experiments.e3 import frozen_controls
s3 = save(ev, e3_model, str(REPORT))
baseline_evaluations, baseline_summaries = frozen_controls(ev, "private", REPORT)
summaries = {}
baseline_evaluations = {}
for label, uri in {"E1": BASELINE_MODEL, "E2": BASE_MODEL}.items():
    baseline = base_model if label == "E2" else load_model(uri)
    assert baseline.golden_ok and baseline.sha256["encoder.onnx"] == ENCODER_SHA256
    if label == "E1":
        assert baseline.pipeline.detector is None
    baseline_ev = evaluate(ds, baseline.pipeline, zero_shot=PROTOCOL == "zero-shot")
    assert [(r["query"], r["animal"], r["gallery_animals"]) for r in baseline_ev.result["test"]["per_query"]] == [
        (r["query"], r["animal"], r["gallery_animals"]) for r in ev.result["test"]["per_query"]]
    baseline_evaluations[label] = baseline_ev
    baseline_summaries[label] = save(baseline_ev, baseline, str(REPORT / f"baseline_{label.lower()}"))
summaries = {**baseline_summaries, "E3": s3}
lines = ["| Model | Scope | Queries | Rank-1 | Rank-5 | Rank-10 | mAP |", "|---|---|---:|---:|---:|---:|---:|"]
for label, summary in summaries.items():
    for scope in ("global", "cat", "dog"):
        values = summary["global"] if scope == "global" else summary["per_species"][scope]
        lines.append(f"| {label} | {scope} | {values['queries']} | {values['rank1']:.2%} | {values['rank5']:.2%} | {values['rank10']:.2%} | {values['mAP']:.4f} |")
display(Markdown("\n".join(lines)))
eer_lines = ["| Model | Descriptive EER | Threshold fitted on these pairs |", "|---|---:|---:|"]
for label, summary in summaries.items():
    eer_lines.append(f"| {label} | {summary['eer']:.2%} | {summary['threshold']:.4f} |")
display(Markdown("\n".join(eer_lines)))
comparison = {label: {key: s3["global"][key] - summary["global"][key]
                     for key in ("rank1", "rank5", "mAP")} for label, summary in baseline_summaries.items()}
(REPORT / "comparison.json").write_text(json.dumps(comparison, indent=2))


#### 3.2.1 Variación por identidad

El promedio global pondera por consultas. La tabla siguiente contrasta las mismas fotografías para cada individuo, conserva su especie y presenta Rank-1/mAP de las tres condiciones. El promedio macro por identidad asigna el mismo peso a cada animal; permite examinar si el resultado global está dominado por mascotas con más fotos. Este análisis describe heterogeneidad y no identifica causas del error.


In [ ]:
# Reuse the same per-query outputs, without additional model inference.
per_model = {label: evaluation.result["test"]["per_query"]
             for label, evaluation in {**baseline_evaluations, "E3": ev}.items()}
identity_findings = []
identity_lines = ["| Identidad | Especie | Consultas | Rank-1 E1 | Rank-1 E2 | Rank-1 E3 | mAP E3 | Δ R1 E3−E2 (pp) |",
                  "|---|---|---:|---:|---:|---:|---:|---:|"]
for animal in sorted({row["animal"] for row in per_model["E3"]}):
    selected = {label: [row for row in rows if row["animal"] == animal]
                for label, rows in per_model.items()}
    values = {label: {"rank1": float(np.mean([row["rank"] == 1 for row in rows])),
                      "mAP": float(np.mean([row["ap"] for row in rows]))}
              for label, rows in selected.items()}
    finding = {"animal": animal, "species": ds.animals[animal]["species"],
               "queries": len(selected["E3"]), "models": values,
               "rank1_delta_e3_e2": values["E3"]["rank1"] - values["E2"]["rank1"]}
    identity_findings.append(finding)
    identity_lines.append(f"| `{animal}` | {finding['species']} | {finding['queries']} | "
                          f"{values['E1']['rank1']:.1%} | {values['E2']['rank1']:.1%} | "
                          f"{values['E3']['rank1']:.1%} | {values['E3']['mAP']:.4f} | "
                          f"{finding['rank1_delta_e3_e2']*100:+.2f} |")
display(Markdown("\n".join(identity_lines)))
macro_rank1 = {label: float(np.mean([row["models"][label]["rank1"] for row in identity_findings]))
               for label in per_model}
print("Rank-1 macro por identidad:", macro_rank1)
analysis = {"dataset": ds.name, "dataset_commit": ds.commit, "dataset_fingerprint": ds.fingerprint,
            "protocol": PROTOCOL, "per_animal": identity_findings, "macro_rank1": macro_rank1,
            "comparison": comparison, "training": train_summary,
            "training_dataset_fingerprint": prepared.fingerprint, "coverage": coverage_metrics}
(REPORT / "analysis.json").write_text(json.dumps(analysis, indent=2))


### 3.3 Discusión y limitaciones

La corrida histórica del piloto aporta evidencia descriptiva favorable a la adaptación métrica en esta muestra: E3 supera a E1/E2 en los indicadores globales reportados. La mejora de Rank-1 se concentra principalmente en gatos; en perros, la mejora de mAP es mayor que el cambio del primer candidato. El resultado no demuestra una mejora uniforme para todas las identidades.

La pérdida de entrenamiento continúa disminuyendo después del mejor checkpoint sin una mejora sostenida de validación. Ese patrón es compatible con sobreajuste a las identidades de entrenamiento, aunque una sola corrida no identifica su causa. La parada temprana conserva el checkpoint seleccionado por validación pública; v4 no interviene en esa decisión.

**Validez interna.** Las etiquetas públicas proceden de los publicadores y mantienen revisión exploratoria; deduplicación y filtro YOLO no verifican cada individuo ni detectan todos los animales presentes. El control de solapamiento público/privado utiliza hashes exactos y no descarta todas las copias transformadas. Solo se ejecutó una semilla.

**Validez externa.** v4 incluye 31 mascotas y ya fue observado en E1/E2. La evaluación no separa sesiones ni demuestra transferencia a nuevos entornos de captura. El porcentaje de fuente usado mide fotos antes de limpieza; no implica que el resto sea elegible ni que aumentar el volumen produzca una mejora proporcional.

**Validez de la medición.** El ranking se condiciona a la especie anotada y a una identidad presente en la galería. Rank-k se promedia por consultas, no por animales; mAP ordena fotos. La EER se ajusta sobre los mismos pares reportados y no valida un umbral de despliegue.

La siguiente fase deberá contrastar más identidades y sesiones, repetir semillas y estimar incertidumbre por identidad, manteniendo un benchmark adicional independiente para cualquier selección posterior de modelos o hiperparámetros.


In [ ]:
captions = {
    "rank1_rank2_map": "Rank-1/Rank-2 por identidad y mAP por fotografía, desglosados por especie; gráficos calculados con los resultados de esta investigación.",
    "confusion": "Filas: identidad real; columnas: primera identidad recuperada. Las confusiones localizan casos, sin explicar su causa.",
    "rank1_por_animal": "Rank-1 por identidad: muestra cómo se distribuyen los aciertos entre mascotas.",
    "distribucion_scores": "Scores positivos y negativos. La EER y su umbral se estiman sobre estos mismos pares; no son calibración independiente.",
    "cmc": "CMC: fracción de consultas cuya identidad correcta aparece entre los primeros k candidatos.",
    "matriz_similitud": "Similitud entre embeddings; las agrupaciones deben interpretarse junto a etiquetas y confusiones.",
}
for name, caption in captions.items():
    image_path = REPORT / f"{name}.png"
    if image_path.exists():
        display(Image(str(image_path), width=760))
        display(Markdown(caption))
findings = ["# E3 · Hallazgos de la ejecución", "",
            f"Dataset: {ds.name}; commit: {ds.commit}.",
            f"Muestra solicitada: {SOURCE_SAMPLE_PCT}% de cada fuente; adquiridas: {coverage_metrics['dataset.sampled_photos']:,} fotos.",
            f"Train efectivo: {coverage_metrics['dataset.train_photos']:,} fotos, {coverage_metrics['dataset.train_identities']:,} identidades; {coverage_metrics['dataset.train_pct']:.4f}% de las fuentes.",
            f"Checkpoint: época {train_summary['best_epoch']}, mAP macro público {train_summary['best_val_mAP']:.4f}.", ""]
for label, delta in comparison.items():
    findings.append(f"- E3 − {label}: Rank-1 {delta['rank1']*100:+.2f} pp; Rank-5 {delta['rank5']*100:+.2f} pp; mAP {delta['mAP']:+.4f}.")
for label, delta in public_deltas.items():
    findings.append(f"- Test público E3 − {label}: Rank-1 {delta['rank1']*100:+.2f} pp; mAP {delta['mAP']:+.4f}.")
findings.extend(["", "Contraste descriptivo; no establece significancia, efecto causal del fondo, ni independencia entre sesiones."])
findings_text = "\n".join(findings)
(REPORT / "findings.md").write_text(findings_text)
display(Markdown(findings_text))


### 3.4 Trazabilidad y publicación

Los artefactos se conservan localmente bajo un directorio nuevo por corrida: configuración, procedencia de fuentes, revisión, manifiesto, auditoría, historia/checkpoint, curvas, modelo ONNX, líneas base y resultados. El recibo `publication.json` indica si el resultado se publicó.

Cada corrida debe publicar sus resultados en MLflow. `LOG_TO_MLFLOW = True` expresa esa política; desactivarlo produce un error en la celda final. `REGISTER_E3` controla por separado el registro de una versión del modelo. La publicación utiliza el experimento existente sin promover un alias.

Las hojas de contacto permanecen en el piloto local. No se copian al reporte ni se publican fotos; el manifiesto de evidencia conserva identificadores/rutas y hashes, no los campos personales de las fuentes.

### 3.5 Referencias metodológicas y de datos

- Radford et al. (2021). [CLIP](https://arxiv.org/abs/2103.00020). Encoder visual congelado.
- Hermans, Beyer y Leibe (2017). [In Defense of the Triplet Loss for Person Re-Identification](https://arxiv.org/abs/1703.07737). Referencia del criterio batch-hard; sus resultados en personas no demuestran rendimiento en mascotas.
- Fuentes públicas: [Dogs of the World](https://www.kaggle.com/datasets/lextoumbourou/dogs-world) y [LCW](https://www.kaggle.com/datasets/dseidli/lcwlabeled-cats-in-the-wild), ambas v1.
- Dataset preparado: [PetTrace Training v1](https://huggingface.co/datasets/alessandro54/pettrace-training/tree/v1), con manifiesto, splits, exclusiones y hashes.

- Zheng et al. (2015). [Scalable Person Re-Identification: A Benchmark](https://openaccess.thecvf.com/content_iccv_2015/html/Zheng_Scalable_Person_Re-Identification_ICCV_2015_paper.html). Contexto metodológico de CMC y mAP; el protocolo de esta tesis se define por separado.
- NIST (2022). [A Tale of Two Errors: Measuring Biometric Algorithms](https://www.nist.gov/blogs/taking-measure/tale-two-errors-measuring-biometric-algorithms). Contexto de errores de coincidencia.
- Fuentes del test público: [Dogs of the World](https://www.kaggle.com/datasets/lextoumbourou/dogs-world) y [LCW](https://www.kaggle.com/datasets/dseidli/lcwlabeled-cats-in-the-wild), v1; selección y preparación propias en `pettrace-training@v1`.

Las entradas reutilizables para la bibliografía de la tesis están en [`docs/references.bib`](../docs/references.bib). Las figuras de este cuaderno son cálculos propios; las fuentes externas fundamentan métodos y datos, no las métricas obtenidas por PetTrace.


In [ ]:
if not LOG_TO_MLFLOW:
    raise ValueError("MLflow publication is mandatory for every experiment run")
with report_backup:
    experiment_design = {
        "schema_version": 2, "experiment": EXPERIMENT, "source_sample_pct": SOURCE_SAMPLE_PCT, "dataset_coverage": coverage_metrics, "protocol": PROTOCOL,
        "phases": ["data_preparation", "representation_and_training", "evaluation"],
        "public_benchmark": {"repo": public_ds.repo, "version": public_ds.version,
                             "commit": public_ds.commit, "fingerprint": public_ds.fingerprint, "split": "test"},
        "task": "individual_pet_retrieval", "gallery": "closed_set_same_species",
        "query_unit": "photograph", "rank_aggregation": "max_score_per_identity",
        "map_aggregation": "photograph_ranking", "capture_session_disjoint": False,
        "private_benchmark": {"repo": ds.repo, "version": ds.version, "commit": ds.commit,
                              "fingerprint": ds.fingerprint},
        "model_sha256": e3_model.sha256, "training": asdict(config),
        "private_identities_used_for_training": False, "eer_scope": "same_evaluation_pairs",
        "mlflow_publication_required": True,
    }
    (REPORT / "experiment-design.json").write_text(json.dumps(experiment_design, indent=2))
    (PUBLIC_REPORT / "experiment-design.json").write_text(json.dumps(experiment_design, indent=2))
    # Both domains must carry the same training-volume evidence.
    for filename in ("dataset-coverage.json", "training-config.json", "training-summary.json", "sources.json"):
        shutil.copy2(REPORT / filename, PUBLIC_REPORT / filename)
    save_provenance(REPORT, e3_model)
    receipt = REPORT / "publication.json"
    receipt.write_text(json.dumps({"published": False, "run_id": None}))
    public_run_id = publish_evaluation(
        public_ev, e3_model, PUBLIC_REPORT, SOURCE_SHA256, experiment=EXPERIMENT,
        expected_dataset_version=public_ds.version, expected_protocol="zero-shot")
    (PUBLIC_REPORT / "publication.json").write_text(json.dumps({"published": True, "run_id": public_run_id}))
    run_id = publish_evaluation(ev, e3_model, REPORT, SOURCE_SHA256, experiment=EXPERIMENT,
                                expected_dataset_version=DATASET, expected_protocol="zero-shot")
    # Dataset coverage is published with every E3 run.
    import time
    from mlflow.entities import Metric, Param
    timestamp = int(time.time() * 1000)
    training_metrics = [Metric(key, float(record[key]), timestamp, int(record["epoch"]))
                        for record in train_summary["history"]
                        for key in ("train.loss", "train.active_triplets", "val.macro_species_mAP",
                                    "training.optimizer_steps", "training.examples_seen", "training.unique_photos_seen")]
    training_metrics.extend([
        Metric("training.best_epoch", float(train_summary["best_epoch"]), timestamp, 0),
        Metric("training.best_val_mAP", float(train_summary["best_val_mAP"]), timestamp, 0)])
    training_params = {f"training.{key}": str(value) for key, value in asdict(config).items()}
    training_params.update({"training.dataset_fingerprint": prepared.fingerprint,
                            "training.dataset_source": json.dumps(training_source, sort_keys=True)})
    client = mlflow.MlflowClient()
    for published_id, folder in ((public_run_id, PUBLIC_REPORT), (run_id, REPORT)):
        client.log_batch(
            published_id,
            metrics=training_metrics + [Metric(key, float(value), timestamp, 0) for key, value in coverage_metrics.items()],
            params=[Param(key, str(value)) for key, value in {**coverage_params, **training_params}.items()])
        client.set_tag(published_id, "source_sample_pct", str(SOURCE_SAMPLE_PCT))
    receipt.write_text(json.dumps({"published": True, "run_id": run_id, "public_run_id": public_run_id}))
    for published_id, folder in ((public_run_id, PUBLIC_REPORT), (run_id, REPORT)):
        if client.get_run(published_id).info.status != "FINISHED":
            raise RuntimeError(f"Publication is incomplete: {published_id}")
        client.log_artifact(published_id, str(folder / "publication.json"), artifact_path="report")
    print("MLflow private run:", run_id, "| Public test run:", public_run_id)
    if REGISTER_E3:
        version = mlflow.register_model(f"runs:/{run_id}/report/model", "pettrace-embedder")
        client = mlflow.MlflowClient()
        client.set_model_version_tag("pettrace-embedder", version.version, "experiment", "E3")
        client.set_model_version_tag("pettrace-embedder", version.version, "status", "candidate")
        print("Registered E3 version:", version.version)
